In [1]:
%pip install sentence-transformers rank-bm25 -q

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import os
import re
import json
import pickle
import logging
import warnings

import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.metrics.pairwise import (
    cosine_similarity
)

warnings.filterwarnings("ignore")

In [3]:
# =========================================================
# DIRECTORY CONFIGURATION
# =========================================================

ROOT_DIR = "/content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan"

DATA_PATH = os.path.join(
    ROOT_DIR,
    "data",
    "processed",
    "cases.csv"
)

RESULT_DIR = os.path.join(
    ROOT_DIR,
    "data",
    "results"
)

LOG_DIR = os.path.join(
    ROOT_DIR,
    "logs"
)

for path in [

    RESULT_DIR,

    LOG_DIR
]:
    os.makedirs(
        path,
        exist_ok=True
    )

print(DATA_PATH)

/content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan/data/processed/cases.csv


In [4]:
# =========================================================
# LOGGING
# =========================================================

LOG_PATH = os.path.join(
    LOG_DIR,
    "solution_reuse.log"
)

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(

    filename=LOG_PATH,

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s",

    filemode="w"
)

logging.info(
    "===== NOTEBOOK 04 STARTED ====="
)

print(LOG_PATH)

/content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan/logs/solution_reuse.log


In [5]:
# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv(DATA_PATH)

print(df.shape)

required_columns = [

    "case_id",

    "amar_putusan",

    "dakwaan",

    "ringkasan_fakta",

    "retrieval_topic"
]

missing_columns = [

    col

    for col in required_columns

    if col not in df.columns
]

if len(missing_columns) > 0:

    raise ValueError(
        f"Kolom tidak ditemukan: {missing_columns}"
    )

print("Validation Passed")

logging.info(
    f"Dataset Loaded : {df.shape}"
)

(30, 12)
Validation Passed


In [6]:
# =========================================================
# TEXT PREPROCESSING
# =========================================================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r'[^a-zA-Z0-9\s]',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()


df["combined_text"] = (

    df["retrieval_topic"].fillna("") + " " +

    df["ringkasan_fakta"].fillna("") + " " +

    df["pasal"].fillna("") + " " +

    df["jenis_putusan"].fillna("") + " " +

    df["dakwaan"].fillna("")
)

df["clean_text"] = (

    df["combined_text"]
    .apply(clean_text)
)

df = df[
    df["clean_text"].str.len() > 20
]

df = df.reset_index(drop=True)

print(df.shape)

logging.info(
    f"Valid Documents : {len(df)}"
)

(30, 14)


In [7]:
# =========================================================
# RETRIEVAL INITIALIZATION
# =========================================================

tfidf = TfidfVectorizer(

    max_features=10000,

    ngram_range=(1,2)
)

tfidf_matrix = tfidf.fit_transform(
    df["clean_text"]
)

tokenized_corpus = [

    doc.split()

    for doc in df["clean_text"]
]

bm25 = BM25Okapi(
    tokenized_corpus
)

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embeddings = embedding_model.encode(

    df["clean_text"].tolist(),

    show_progress_bar=True
)

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Retrieval Ready")

logging.info(
    "Retrieval Components Loaded"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieval Ready


In [8]:
# =========================================================
# INDIVIDUAL RETRIEVAL METHODS
# =========================================================

def retrieve_tfidf(
    query,
    k=5
):

    query = clean_text(query)

    query_vec = tfidf.transform(
        [query]
    )

    scores = cosine_similarity(
        query_vec,
        tfidf_matrix
    ).flatten()

    top_idx = scores.argsort()[::-1][:k]

    results = []

    for idx in top_idx:

        results.append({

            "case_id":
            df.iloc[idx]["case_id"],

            "similarity":
            float(scores[idx])
        })

    return pd.DataFrame(results)


def retrieve_bm25(
    query,
    k=5
):

    query = clean_text(query)

    scores = bm25.get_scores(
        query.split()
    )

    top_idx = np.argsort(scores)[::-1][:k]

    results = []

    for idx in top_idx:

        results.append({

            "case_id":
            df.iloc[idx]["case_id"],

            "similarity":
            float(scores[idx])
        })

    return pd.DataFrame(results)


def retrieve_bert(
    query,
    k=5
):

    query = clean_text(query)

    query_embedding = embedding_model.encode(
        [query]
    )

    scores = cosine_similarity(
        query_embedding,
        embeddings
    ).flatten()

    top_idx = scores.argsort()[::-1][:k]

    results = []

    for idx in top_idx:

        results.append({

            "case_id":
            df.iloc[idx]["case_id"],

            "similarity":
            float(scores[idx])
        })

    return pd.DataFrame(results)

In [9]:
# =========================================================
# HYBRID RETRIEVAL
# =========================================================

query_dictionary = {

    "bahan peledak":
    "bom ikan peledak potasium",

    "pukat harimau":
    "trawl alat tangkap",

    "tanpa surat izin penangkapan ikan":
    "sipi izin penangkapan ikan",

    "tanpa surat izin usaha perikanan":
    "siup izin usaha perikanan",

    "tanpa surat persetujuan berlayar":
    "spb surat persetujuan berlayar",

    "kapal asing":
    "vietnam thailand china berbendera asing"
}


def expand_query(query):

    query = query.lower()

    expanded = query

    for key, value in query_dictionary.items():

        if key in query:

            expanded += " " + value

    return expanded


def retrieve_hybrid(
    query,
    k=5
):

    query = expand_query(query)

    query = clean_text(query)

    query_tfidf = tfidf.transform(
        [query]
    )

    tfidf_scores = cosine_similarity(
        query_tfidf,
        tfidf_matrix
    ).flatten()

    bm25_scores = bm25.get_scores(
        query.split()
    )

    query_embedding = embedding_model.encode(
        [query]
    )

    bert_scores = cosine_similarity(
        query_embedding,
        embeddings
    ).flatten()

    tfidf_norm = (
        tfidf_scores - tfidf_scores.min()
    ) / (
        tfidf_scores.max() -
        tfidf_scores.min() +
        1e-10
    )

    bm25_norm = (
        bm25_scores -
        bm25_scores.min()
    ) / (
        bm25_scores.max() -
        bm25_scores.min() +
        1e-10
    )

    bert_norm = (
        bert_scores -
        bert_scores.min()
    ) / (
        bert_scores.max() -
        bert_scores.min() +
        1e-10
    )

    final_scores = (

        0.20 * tfidf_norm +

        0.35 * bm25_norm +

        0.45 * bert_norm
    )

    top_idx = final_scores.argsort()[::-1][:10]

    pairs = [

        (query, df.iloc[i]["clean_text"])

        for i in top_idx
    ]

    rerank_scores = cross_encoder.predict(
        pairs
    )

    reranked = sorted(

        zip(top_idx, rerank_scores),

        key=lambda x: x[1],

        reverse=True
    )

    results = []

    for idx, score in reranked[:k]:

        results.append({

            "case_id":
            df.iloc[idx]["case_id"],

            "similarity":
            float(score)
        })

    return pd.DataFrame(results)

In [10]:
# =========================================================
# CASE SOLUTIONS
# =========================================================

case_solutions = {}

for _, row in df.iterrows():

    case_id = row["case_id"]

    if pd.notna(row["amar_putusan"]) and str(row["amar_putusan"]).strip() != "":
        solution = str(row["amar_putusan"])

    elif pd.notna(row["dakwaan"]) and str(row["dakwaan"]).strip() != "":
        solution = str(row["dakwaan"])

    else:
        solution = "Tidak tersedia"

    case_solutions[case_id] = solution

print("Jumlah solusi :", len(case_solutions))

logging.info(
    f"Case Solutions : {len(case_solutions)}"
)

Jumlah solusi : 30


In [11]:
# =========================================================
# MAJORITY VOTE
# =========================================================

from collections import Counter

def predict_outcome_majority(
    query,
    k=5
):

    retrieved = retrieve_hybrid(
        query,
        k
    )

    solutions = []

    for case_id in retrieved["case_id"]:

        solutions.append(
            case_solutions[case_id]
        )

    prediction = Counter(
        solutions
    ).most_common(1)[0][0]

    return prediction

In [12]:
# =========================================================
# WEIGHTED SIMILARITY
# =========================================================

from collections import defaultdict

def predict_outcome_weighted(
    query,
    k=5
):

    retrieved = retrieve_hybrid(
        query,
        k
    )

    score_map = defaultdict(float)

    for _, row in retrieved.iterrows():

        solution = case_solutions[
            row["case_id"]
        ]

        similarity = row["similarity"]

        score_map[
            solution
        ] += similarity

    prediction = max(
        score_map,
        key=score_map.get
    )

    return prediction

In [13]:
# =========================================================
# PREDICT OUTCOME
# =========================================================

def predict_outcome(
    query,
    method="weighted",
    k=5
):

    if method == "majority":

        return predict_outcome_majority(
            query,
            k
        )

    return predict_outcome_weighted(
        query,
        k
    )

In [14]:
# =========================================================
# DEMO MANUAL
# =========================================================

print("=" * 100)
print("DEMO CASE SOLUTION REUSE")
print("=" * 100)

demo_cases = [

    {
        "query": "penangkapan ikan menggunakan bahan peledak",
        "actual_solution": "Bom Ikan"
    },

    {
        "query": "penggunaan alat tangkap trawl",
        "actual_solution": "Trawl"
    },

    {
        "query": "kapal asing melakukan penangkapan ikan",
        "actual_solution": "Kapal Asing"
    },

    {
        "query": "penangkapan ikan tanpa SIPI",
        "actual_solution": "Tanpa SIPI"
    },

    {
        "query": "kapal tidak memiliki surat persetujuan berlayar",
        "actual_solution": "Tanpa SPB"
    }

]

for i, demo in enumerate(demo_cases, start=1):

    prediction = predict_outcome(
        demo["query"],
        method="weighted"
    )

    retrieved = retrieve_hybrid(
        demo["query"],
        k=5
    )

    print("\n" + "="*80)

    print(f"Demo {i}")

    print("="*80)

    print("Query :")
    print(demo["query"])

    print("\nPredicted Solution :")
    print(prediction[:500])

    print("\nExpected Category :")
    print(demo["actual_solution"])

    print("\nTop 5 Case IDs :")
    print(retrieved["case_id"].tolist())

DEMO CASE SOLUTION REUSE


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Demo 5
Query :
kapal tidak memiliki surat persetujuan berlayar

Predicted Solution :
Menyatakan Terdakwa MULIADI Alias MOLET terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana “Turut serta melakukan perbuatan dengan sengaja memasukkan, mengeluarkan, mengadakan, mengedarkan dan/atau memelihara ikan yang merugikan masyarakat, pembudidayaan ikan, sumber daya ikan, dan atau lingkungan sumber daya ikan ke dalam dan atau ke luar Wilayah Pengelolaan Perikanan Republik Indonesia sebagaimana dimaksud dalam Pasal 16 ayat (1) yang dilakukan secara berlanjut” sebagaimana di

Expected Category :
Tanpa SPB

Top 5 Case IDs :
['case_015', 'case_007', 'case_009', 'case_013', 'case_004']


In [15]:
# =========================================================
# GENERATE PREDICTIONS
# =========================================================

prediction_results = []

for idx, row in df.iterrows():

    query = row["retrieval_topic"]

    retrieved = retrieve_hybrid(
        query,
        k=5
    )

    prediction = predict_outcome(
        query,
        method="weighted"
    )

    similarity_scores = ";".join(

        retrieved["similarity"]
        .round(4)
        .astype(str)
        .tolist()

    )

    top_cases = ";".join(

        retrieved["case_id"]
        .astype(str)
        .tolist()

    )

    prediction_results.append({

        "query_id":
        idx + 1,

        "query":
        query,

        "actual_solution":
        case_solutions[row["case_id"]],

        "predicted_solution":
        prediction,

        "top_5_case_ids":
        top_cases,

        "top_5_similarity":
        similarity_scores

    })

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
# =========================================================
# SAVE PREDICTIONS
# =========================================================

output_file = os.path.join(

    RESULT_DIR,

    "predictions.csv"
)

pd.DataFrame(
    prediction_results
).to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"
)

saved_files = [output_file]

print(
    f"Saved : {output_file}"
)

logging.info(
    f"Prediction Saved : {output_file}"
)

Saved : /content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan/data/results/predictions.csv


In [17]:
# =========================================================
# RETAIN CASE BASE
# =========================================================

retain_file = os.path.join(

    RESULT_DIR,

    "case_base_retained.csv"

)

retain_df = pd.DataFrame({

    "case_id":
    df["case_id"],

    "retrieval_topic":
    df["retrieval_topic"],

    "amar_putusan":
    df["amar_putusan"],

    "dakwaan":
    df["dakwaan"],

    "clean_text":
    df["clean_text"],

    "combined_text":
    df["combined_text"]

})

retain_df.to_csv(

    retain_file,

    index=False,

    encoding="utf-8-sig"

)

print()

print("Case Base Retained")

print(retain_df.shape)

display(
    retain_df.head()
)

logging.info(

    f"Retain File Saved : {retain_file}"

)


Case Base Retained
(30, 6)


,case_id,retrieval_topic,amar_putusan,dakwaan,clean_text,combined_text
0,case_001,Bom Ikan,"mengadili,“Mereka yang melakukan, yang menyuru...",didakwa:KESATU :BahwaTerdakwaISOPIAN binRABBIL...,bom ikan pada hari senin tanggal 02 nopember 2...,Bom Ikan pada hari senin tanggal 02 nopember 2...
1,case_002,Trawl,MENGADILI SENDIRI:1.Menyatakan Terdakwa Rizal ...,didakwa dengan dakwaan sebagai berikut:Dakwaan...,trawl bahwa terdakwa melanggar pasal 98 undang...,Trawl bahwa terdakwa melanggar pasal 98 undang...
2,case_003,Trawl,mengadili Pengadilan NegeriPalembang dikarenak...,didakwa dengan dakwaan sebagai berikut :Bahwa ...,trawl pada hari kamis tanggal 4 februari 2016 ...,"Trawl pada hari kamis,tanggal 4 februari 2016 ..."
3,case_004,Tanpa SIPI,mengadili Perkara Pidana dalam tingkat banding...,didakwa sebagai berikut : Kesatu: Bahwa terdak...,tanpa sipi bahwa terdakwa sugiyantoro als giya...,Tanpa SIPI bahwa terdakwa sugiyantoro als. giy...
4,case_005,Trawl,mengadili sendiri perkara inidengan amar putus...,didakwa dengan dakwaan sebagai berikut:-Kesatu...,trawl bahwa terdakwa mengakuibelumpernah dihuk...,Trawl bahwa terdakwa mengakuibelumpernah dihuk...


In [18]:
# =========================================================
# FINAL LOG
# =========================================================

logging.info(
    "===== NOTEBOOK 04 FINISHED ====="
)

logging.shutdown()

print("=" * 60)
print("NOTEBOOK 04 COMPLETED")
print("=" * 60)

print()

for file in saved_files:

    print(file)

print()

print(
    "Jumlah Prediksi :",
    len(prediction_results)
)

NOTEBOOK 04 COMPLETED

/content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan/data/results/predictions.csv

Jumlah Prediksi : 30
